# Data Understanding 4: `previous_application`

Every earlier application each customer made **to Home Credit itself**: approved, refused, cancelled or unused. It is the parent of `installments_payments`: each previous loan with payments started as a row here.

Each table is profiled with the same five questions:

1. **Grain:** what does one row represent?
2. **Real rows:** what does the data actually look like?
3. **Types:** what type is each column, and what does it really mean?
4. **Missing data:** where are the blanks, and why?
5. **Ranges:** do the values make sense, and what looks wrong?


## Setup

DuckDB queries the CSV files directly, so no table is loaded fully into memory. Only small, aggregated results are passed to pandas for display. Run the cells top to bottom.

In [1]:
import duckdb
import pandas as pd

PREV = 'data/raw/previous_application.csv'

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)

---
## Q1. Grain

In [2]:
duckdb.sql(f'''
    SELECT
        COUNT(*)                   AS total_rows,
        COUNT(DISTINCT SK_ID_PREV) AS unique_prev_applications,
        COUNT(DISTINCT SK_ID_CURR) AS unique_customers
    FROM '{PREV}'
''').show()

┌────────────┬──────────────────────────┬──────────────────┐
│ total_rows │ unique_prev_applications │ unique_customers │
│   int64    │          int64           │      int64       │
├────────────┼──────────────────────────┼──────────────────┤
│    1670214 │                  1670214 │           338857 │
└────────────┴──────────────────────────┴──────────────────┘



What happened to each application?

In [3]:
duckdb.sql(f'''
    SELECT
        NAME_CONTRACT_STATUS,
        COUNT(*) AS applications,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{PREV}'
    GROUP BY NAME_CONTRACT_STATUS
    ORDER BY applications DESC
''').show()

┌──────────────────────┬──────────────┬────────┐
│ NAME_CONTRACT_STATUS │ applications │  pct   │
│       varchar        │    int64     │ double │
├──────────────────────┼──────────────┼────────┤
│ Approved             │      1036781 │  62.07 │
│ Canceled             │       316319 │  18.94 │
│ Refused              │       290678 │   17.4 │
│ Unused offer         │        26436 │   1.58 │
└──────────────────────┴──────────────┴────────┘



**Findings**

- 1,670,214 rows = 1,670,214 unique `SK_ID_PREV`: **one row = one previous application**.
- 338,857 customers, about **4.9 applications per customer**. Must be aggregated per `SK_ID_CURR` before joining.
- Statuses: Approved 1,036,781 (62%) · Canceled 316,319 (19%) · Refused 290,678 (17%) · Unused offer 26,436 (1.6%).
- **Link to `installments_payments`:** only ~998k of the 1.67M applications have payments. Refused, cancelled and unused applications never became loans, so they have no repayment history. **No payment history is not a bad sign in itself.**

---
## Q2. Real rows

In [4]:
prev_sample = duckdb.sql(f"SELECT * FROM '{PREV}' LIMIT 3").df()
prev_sample.T

,0,1,2
SK_ID_PREV,2030495,2802425,2523466
SK_ID_CURR,271877,108129,122040
NAME_CONTRACT_TYPE,Consumer loans,Cash loans,Cash loans
AMT_ANNUITY,1730.43,25188.615,15060.735
AMT_APPLICATION,17145.0,607500.0,112500.0
AMT_CREDIT,17145.0,679671.0,136444.5
AMT_DOWN_PAYMENT,0.0,NaN,NaN
AMT_GOODS_PRICE,17145.0,607500.0,112500.0
WEEKDAY_APPR_PROCESS_START,SATURDAY,THURSDAY,TUESDAY
HOUR_APPR_PROCESS_START,15,11,11


One customer with both a refused and an approved application:

In [5]:
duckdb.sql(f'''
    SELECT
        SK_ID_PREV, NAME_CONTRACT_TYPE, NAME_CONTRACT_STATUS,
        AMT_APPLICATION, AMT_CREDIT, AMT_ANNUITY, CNT_PAYMENT,
        DAYS_DECISION, CODE_REJECT_REASON, NAME_YIELD_GROUP
    FROM '{PREV}'
    WHERE SK_ID_CURR = (
        SELECT SK_ID_CURR
        FROM '{PREV}'
        GROUP BY SK_ID_CURR
        HAVING SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END) > 0
           AND SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) > 0
        ORDER BY SK_ID_CURR
        LIMIT 1
    )
    ORDER BY DAYS_DECISION
''').df()

,SK_ID_PREV,NAME_CONTRACT_TYPE,NAME_CONTRACT_STATUS,AMT_APPLICATION,AMT_CREDIT,AMT_ANNUITY,CNT_PAYMENT,DAYS_DECISION,CODE_REJECT_REASON,NAME_YIELD_GROUP
0,2299329,Consumer loans,Approved,26912.34,24219.0,2482.92,12.0,-617,XAP,middle
1,1020698,Cash loans,Approved,454500.00,481495.5,39954.51,18.0,-438,XAP,high
2,2190416,Consumer loans,Approved,334917.00,267930.0,29027.52,12.0,-311,XAP,high
3,2827850,Revolving loans,Canceled,0.00,0.0,NaN,NaN,-181,XAP,XNA
4,2078043,Cash loans,Approved,675000.00,675000.0,24246.00,48.0,-181,XAP,low_normal
5,2545789,Cash loans,Canceled,0.00,0.0,NaN,NaN,-181,XAP,XNA
6,1697039,Cash loans,Refused,688500.00,906615.0,32696.10,48.0,-181,LIMIT,low_normal
7,1489396,Revolving loans,Approved,270000.00,270000.0,13500.00,0.0,-181,XAP,XNA
8,1243599,Cash loans,Canceled,0.00,0.0,NaN,NaN,-181,XAP,XNA


**Key columns**

| Column | Meaning |
|---|---|
| `NAME_CONTRACT_TYPE` | Cash loan, Consumer loan (point of sale), Revolving loan |
| `NAME_CONTRACT_STATUS` | Approved, Refused, Canceled, Unused offer |
| `AMT_APPLICATION` | Amount **asked for** |
| `AMT_CREDIT` | **Final** loan amount worked out for the application (can include extras like insurance; exists on refused rows too) |
| `AMT_ANNUITY` / `CNT_PAYMENT` | Repayment amount / number of instalments |
| `DAYS_DECISION` | When the decision was made |
| `CODE_REJECT_REASON` | Why it was refused |
| `NAME_YIELD_GROUP` | Interest rate band |
| `DAYS_FIRST_DUE` ... `DAYS_TERMINATION` | Repayment start, end and contract close |

**Findings**

- `AMT_CREDIT` is not simply "granted": it can be **higher** than the amount asked (likely extras such as insurance bundled in), and refused applications have one too. Only on approved rows does it approximate money actually lent.
- Asked vs final on approved loans shows how much the lender trimmed or topped up a request.
- `AMT_APPLICATION = 0` appears mostly on cancelled applications: the request was never completed.
- The example customer's small loans were approved while repeated large requests (675k to 990k) were refused: a pattern of asking for more than the lender will give.
- **The same column can mean different things depending on the status**, so check its meaning per status before using it.

---
## Q3. Column types and meaning

In [6]:
duckdb.sql(f'''
    SELECT column_type, COUNT(*) AS num_columns
    FROM (DESCRIBE SELECT * FROM '{PREV}')
    GROUP BY column_type
    ORDER BY num_columns DESC
''').show()

┌─────────────┬─────────────┐
│ column_type │ num_columns │
│   varchar   │    int64    │
├─────────────┼─────────────┤
│ VARCHAR     │          16 │
│ DOUBLE      │          15 │
│ BIGINT      │           6 │
└─────────────┴─────────────┘



In [7]:
for col in ['NAME_CONTRACT_TYPE', 'CODE_REJECT_REASON', 'NAME_YIELD_GROUP']:
    print(col)
    duckdb.sql(f'''
        SELECT {col}, COUNT(*) AS n
        FROM '{PREV}'
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT 10
    ''').show()

NAME_CONTRACT_TYPE
┌────────────────────┬────────┐
│ NAME_CONTRACT_TYPE │   n    │
│      varchar       │ int64  │
├────────────────────┼────────┤
│ Cash loans         │ 747553 │
│ Consumer loans     │ 729151 │
│ Revolving loans    │ 193164 │
│ XNA                │    346 │
└────────────────────┴────────┘

CODE_REJECT_REASON
┌────────────────────┬─────────┐
│ CODE_REJECT_REASON │    n    │
│      varchar       │  int64  │
├────────────────────┼─────────┤
│ XAP                │ 1353093 │
│ HC                 │  175231 │
│ LIMIT              │   55680 │
│ SCO                │   37467 │
│ CLIENT             │   26436 │
│ SCOFR              │   12811 │
│ XNA                │    5244 │
│ VERIF              │    3535 │
│ SYSTEM             │     717 │
└────────────────────┴─────────┘

NAME_YIELD_GROUP
┌──────────────────┬────────┐
│ NAME_YIELD_GROUP │   n    │
│     varchar      │ int64  │
├──────────────────┼────────┤
│ XNA              │ 517215 │
│ middle           │ 385532 │
│ high       

**Findings**

- 37 columns; the text columns are **categories** (contract type, status, reject reason, yield group, channel, product...).
- **`XNA` and `XAP`** appear in many text columns: Home Credit's codes for *not available* and *not applicable*. They are **missing values written as text**, so `IS NULL` does not find them.
- `CODE_REJECT_REASON = 'XAP'` on 1,353,093 rows: every Approved and Canceled application (minus 8 approved with a reason), and 1 refused. Unused offers carry a real reason. XAP = "not refused, so no reason": doesn't apply, written as text.
- `NAME_CONTRACT_TYPE`: Cash 747,553 · Consumer 729,151 · Revolving 193,164 · XNA 346 (0.02%).
- Share of applications with a real reject reason: cash ~22%, consumer ~14%, revolving ~26%. Home Credit refuses cash and revolving requests more often than point-of-sale loans.
- `NFLAG_...` columns are 0/1 flags stored as numbers.

---
## Q4. Missing data

In [8]:
prev_summary = duckdb.sql(f"SUMMARIZE SELECT * FROM '{PREV}'").df()
prev_summary[['column_name', 'column_type', 'null_percentage']].sort_values('null_percentage', ascending=False)

,column_name,column_type,null_percentage
14,RATE_INTEREST_PRIVILEGED,DOUBLE,99.64
13,RATE_INTEREST_PRIMARY,DOUBLE,99.64
6,AMT_DOWN_PAYMENT,DOUBLE,53.64
12,RATE_DOWN_PAYMENT,DOUBLE,53.64
20,NAME_TYPE_SUITE,VARCHAR,49.12
35,DAYS_TERMINATION,DOUBLE,40.30
31,DAYS_FIRST_DRAWING,DOUBLE,40.30
32,DAYS_FIRST_DUE,DOUBLE,40.30
33,DAYS_LAST_DUE_1ST_VERSION,DOUBLE,40.30
34,DAYS_LAST_DUE,DOUBLE,40.30


Are repayment fields blank mostly for applications that never became loans?

In [9]:
duckdb.sql(f'''
    SELECT
        NAME_CONTRACT_STATUS,
        COUNT(*) AS applications,
        ROUND(100.0 * AVG(CASE WHEN AMT_ANNUITY IS NULL THEN 1 ELSE 0 END), 2) AS pct_annuity_blank,
        ROUND(100.0 * AVG(CASE WHEN CNT_PAYMENT IS NULL THEN 1 ELSE 0 END), 2) AS pct_cnt_payment_blank
    FROM '{PREV}'
    GROUP BY NAME_CONTRACT_STATUS
    ORDER BY applications DESC
''').show()

┌──────────────────────┬──────────────┬───────────────────┬───────────────────────┐
│ NAME_CONTRACT_STATUS │ applications │ pct_annuity_blank │ pct_cnt_payment_blank │
│       varchar        │    int64     │      double       │        double         │
├──────────────────────┼──────────────┼───────────────────┼───────────────────────┤
│ Approved             │      1036781 │               0.0 │                   0.0 │
│ Canceled             │       316319 │             96.68 │                 96.68 │
│ Refused              │       290678 │             14.07 │                 14.07 │
│ Unused offer         │        26436 │             96.55 │                 96.55 │
└──────────────────────┴──────────────┴───────────────────┴───────────────────────┘



Missing values hidden as text:

In [10]:
duckdb.sql(f'''
    SELECT
        ROUND(100.0 * AVG(CASE WHEN NAME_CONTRACT_TYPE = 'XNA' THEN 1 ELSE 0 END), 3) AS pct_contract_type_xna,
        ROUND(100.0 * AVG(CASE WHEN CODE_REJECT_REASON = 'XAP' THEN 1 ELSE 0 END), 2) AS pct_reject_reason_xap,
        ROUND(100.0 * AVG(CASE WHEN NAME_YIELD_GROUP   = 'XNA' THEN 1 ELSE 0 END), 2) AS pct_yield_group_xna
    FROM '{PREV}'
''').show()

┌───────────────────────┬───────────────────────┬─────────────────────┐
│ pct_contract_type_xna │ pct_reject_reason_xap │ pct_yield_group_xna │
│        double         │        double         │       double        │
├───────────────────────┼───────────────────────┼─────────────────────┤
│                 0.021 │                 81.01 │               30.97 │
└───────────────────────┴───────────────────────┴─────────────────────┘



**Findings**

Columns with identical blank percentages go missing together, and almost every group is explained by **another column**:

| Group | Columns | Blank | Explained by | Reading |
|---|---|---|---|---|
| A | `RATE_INTEREST_PRIMARY`, `RATE_INTEREST_PRIVILEGED` | 99.64% | Present on only 5,951 approved loans (0.6%) | Truly unknown: **drop** |
| B | `AMT_DOWN_PAYMENT`, `RATE_DOWN_PAYMENT` | 53.64% | **Loan type:** consumer 0.7%, cash 93.6%, revolving 98.8% blank | Doesn't apply to cash and revolving loans |
| C | `DAYS_FIRST_DRAWING` ... `DAYS_TERMINATION`, `NFLAG_INSURED_ON_APPROVAL` | 40.30% | **Status:** approved 3.8%, all others 100% | Doesn't apply when no loan was made |
| D | `AMT_ANNUITY`, `CNT_PAYMENT` | 22.29% | **Status:** approved 0%, canceled 96.7%, unused 96.6%, refused 14.1% | Doesn't apply when no loan was made |

- The 3.8% of approved loans missing repayment dates (~39,600) are spread across types (cash 7.6%, revolving 4.1%, consumer 1.9%), so they are not a revolving-loan effect. Most likely approved but never drawn.
- Refused applications mostly keep annuity and payment count: loan terms are worked out before the decision.
- **Hidden missing values:** `SUMMARIZE` reports `CODE_REJECT_REASON` and `NAME_YIELD_GROUP` as 0% missing, yet they are **81.0% XAP** and **31.0% XNA**. A NULL check alone overstates completeness.

---
## Q5. Ranges and anomalies

In [11]:
prev_summary[prev_summary['column_type'] != 'VARCHAR'][['column_name', 'min', 'max', 'avg']]

,column_name,min,max,avg
0,SK_ID_PREV,1000001,2845382,1923089.1353305625
1,SK_ID_CURR,100001,456255,278357.1740986484
3,AMT_ANNUITY,0.0,418058.145,15955.120659452143
4,AMT_APPLICATION,0.0,6905160.0,175233.8603604295
5,AMT_CREDIT,0.0,6905160.0,196114.02121797943
6,AMT_DOWN_PAYMENT,-0.9,3060045.0,6697.402138796695
7,AMT_GOODS_PRICE,0.0,6905160.0,227847.2792833453
9,HOUR_APPR_PROCESS_START,0,23,12.484181667738385
11,NFLAG_LAST_APPL_IN_DAY,0,1,0.9964675185335532
12,RATE_DOWN_PAYMENT,-1.4978763414307847e-05,1.0,0.07963681542840366


The `365243` placeholder from `application_train`:

In [12]:
duckdb.sql(f'''
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN DAYS_FIRST_DRAWING        = 365243 THEN 1 ELSE 0 END) AS first_drawing_365243,
        SUM(CASE WHEN DAYS_FIRST_DUE            = 365243 THEN 1 ELSE 0 END) AS first_due_365243,
        SUM(CASE WHEN DAYS_LAST_DUE_1ST_VERSION = 365243 THEN 1 ELSE 0 END) AS last_due_1st_365243,
        SUM(CASE WHEN DAYS_LAST_DUE             = 365243 THEN 1 ELSE 0 END) AS last_due_365243,
        SUM(CASE WHEN DAYS_TERMINATION          = 365243 THEN 1 ELSE 0 END) AS termination_365243
    FROM '{PREV}'
''').show()

┌─────────┬──────────────────────┬──────────────────┬─────────────────────┬─────────────────┬────────────────────┐
│  total  │ first_drawing_365243 │ first_due_365243 │ last_due_1st_365243 │ last_due_365243 │ termination_365243 │
│  int64  │        int128        │      int128      │       int128        │     int128      │       int128       │
├─────────┼──────────────────────┼──────────────────┼─────────────────────┼─────────────────┼────────────────────┤
│ 1670214 │               934444 │            40645 │               93864 │          211221 │             225913 │
└─────────┴──────────────────────┴──────────────────┴─────────────────────┴─────────────────┴────────────────────┘



Zero and negative amounts by status:

In [13]:
duckdb.sql(f'''
    SELECT
        NAME_CONTRACT_STATUS,
        COUNT(*) AS applications,
        SUM(CASE WHEN AMT_CREDIT = 0 THEN 1 ELSE 0 END) AS credit_zero,
        SUM(CASE WHEN AMT_CREDIT < 0 OR AMT_APPLICATION < 0 OR AMT_ANNUITY < 0 THEN 1 ELSE 0 END) AS any_negative
    FROM '{PREV}'
    GROUP BY NAME_CONTRACT_STATUS
    ORDER BY applications DESC
''').show()

┌──────────────────────┬──────────────┬─────────────┬──────────────┐
│ NAME_CONTRACT_STATUS │ applications │ credit_zero │ any_negative │
│       varchar        │    int64     │   int128    │    int128    │
├──────────────────────┼──────────────┼─────────────┼──────────────┤
│ Approved             │      1036781 │        1551 │            0 │
│ Canceled             │       316319 │      305566 │            0 │
│ Refused              │       290678 │       29159 │            0 │
│ Unused offer         │        26436 │         492 │            0 │
└──────────────────────┴──────────────┴─────────────┴──────────────┘



**Findings**

- `DAYS_DECISION` reaches back -2,922 days (8 years), consistent with the other tables.
- **365243 placeholder** is the maximum of all five repayment date columns and appears only on approved loans. It encodes "has not happened" (e.g. not terminated, never drawn), not a real date. `DAYS_FIRST_DRAWING` averages 342,209, so almost all its filled values are the placeholder: an average is meaningless until placeholders are removed. Treatment: set to NULL, keep the meaning as a flag.
- `AMT_DOWN_PAYMENT` min -0.9 and `RATE_DOWN_PAYMENT` min -0.000015: negative but effectively zero (rounding). Treated as 0.
- `SELLERPLACE_AREA` is an anonymised location code: -1 = unknown, and its average has no meaning.
- `CNT_PAYMENT` up to 84 instalments (7 years).
- **This table hides missing data four ways:** NULL, `XNA`/`XAP` text, the 365243 placeholder and -1. Each needs explicit handling before building features.

---
## Summary

| Question | `previous_application` |
|---|---|
| Grain | One row per previous application (1.67M), ~4.9 per customer |
| Real rows | Asked vs granted amount; refusals carry reject reasons |
| Types | Categorical text columns; `XNA`/`XAP` are missing values as text |
| Missing | Repayment fields blank where no loan was made (doesn't apply) |
| Anomalies | 365243 placeholder in repayment date columns |

**Carried into feature engineering:** aggregate per `SK_ID_CURR` (number of applications, refusal count and rate, approved amounts, asked vs final ratio on approved loans); convert `XNA`/`XAP` and 365243 to NULL; do not treat absent payment history as a risk signal on its own.